# 15 -  Export GGUF Model

### Setup

In [1]:
# ==== SETUP ====
!pip install --no-deps bitsandbytes accelerate xformers peft trl triton unsloth_zoo
!pip install --no-deps cut_cross_entropy unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 57.4 MB/s eta 0:00:00


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Konfigurasi

In [3]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_Faiss")
    except ImportError:
        token = os.environ.get("HF_Faiss")
        if token is None:
            raise RuntimeError("HF_TOKEN tidak ditemukan.")
        return token


HF_TOKEN = get_hf_token()
HF_USER = "Makaareeem"

CANDIDATES = [
    {
        "key": "llama3.2",
        "base_model": "unsloth/Llama-3.2-3B-Instruct",
        "adapter_repo": f"{HF_USER}/llama3.2-qlora-adapter",
        "max_seq_length": 1536,
        "system_prompt": (
            "Anda adalah asisten yang menjawab pertanyaan seputar publikasi "
            "statistik BPS berdasarkan konteks yang diberikan. Kutip bagian "
            "konteks yang relevan sebelum menjawab."
        ),
        "repeat_penalty": 1.0,
    },
    {
        "key": "gemma2",
        "base_model": "unsloth/gemma-2-2b-it",
        "adapter_repo": f"{HF_USER}/gemma2-qlora-adapter",
        "max_seq_length": 1024,
        "system_prompt": (
            "Anda adalah asisten yang menjawab pertanyaan seputar publikasi "
            "statistik BPS berdasarkan konteks yang diberikan. Kutip bagian "
            "konteks yang relevan sebelum menjawab."
        ),
        "repeat_penalty": 1.0,
    },
]

QUANT_METHODS = ["q4_k_m"]

## 2. Bentuk Model

In [4]:
def build_modelfile(gguf_filename, system_prompt, repeat_penalty):
    return f'''FROM ./{gguf_filename}

PARAMETER temperature 0.1
PARAMETER repeat_penalty {repeat_penalty}
PARAMETER num_ctx 4096

SYSTEM """{system_prompt}"""
'''


def export_gguf(model_name, gguf_repo, tokenizer_source_name, c):
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=c["max_seq_length"],
        dtype=None,
        load_in_4bit=False,
        token=HF_TOKEN,
    )

    for quant in QUANT_METHODS:
        model.push_to_hub_gguf(
            gguf_repo,
            tokenizer,
            quantization_method=quant,
            token=HF_TOKEN,
        )

    modelfile_content = build_modelfile(
        gguf_filename=f"{gguf_repo.split('/')[-1]}.{QUANT_METHODS[0].upper()}.gguf",
        system_prompt=c["system_prompt"],
        repeat_penalty=c["repeat_penalty"],
    )
    modelfile_name = f"{gguf_repo.split('/')[-1]}.Modelfile"
    with open(modelfile_name, "w", encoding="utf-8") as f:
        f.write(modelfile_content)

    from huggingface_hub import HfApi
    HfApi().upload_file(
        path_or_fileobj=modelfile_name,
        path_in_repo="Modelfile",
        repo_id=gguf_repo,
        token=HF_TOKEN,
    )

    del model, tokenizer
    import torch
    torch.cuda.empty_cache()

## 3. Main Program

In [5]:
for c in CANDIDATES:
    export_gguf(
        model_name=c["base_model"],
        gguf_repo=f"{HF_USER}/{c['key']}-base-gguf",
        tokenizer_source_name=c["base_model"],
        c=c,
    )
    print(f"{c['key']} base selesai -> {HF_USER}/{c['key']}-base-gguf")

    export_gguf(
        model_name=c["adapter_repo"],
        gguf_repo=f"{HF_USER}/{c['key']}-finetuned-gguf",
        tokenizer_source_name=c["adapter_repo"],
        c=c,
    )
    print(f"{c['key']} fine-tuned selesai -> {HF_USER}/{c['key']}-finetuned-gguf")

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct as a legacy tokenizer.
Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Converting model to GGUF format...
Unsloth: Model is not a PEFT model. Saving directly without LoRA merge...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Uploading Ollama Modelfile...


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Unsloth: Successfully uploaded GGUF to https://huggingface.co/Makaareeem/gemma2-finetuned-gguf
Unsloth: Cleaning up temporary files...
gemma2 fine-tuned selesai -> Makaareeem/gemma2-finetuned-gguf
